In [ ]:
def estimate_inference_stage(start, W, label, maxfev=MAX_FUNCTION_EVALUATIONS):
    """Nelder-Mead exploration followed by a local simplex refinement of its best trial."""
    W = np.asarray(W, dtype=float)
    if W.shape != (6, 6) or not np.isfinite(W).all():
        raise ValueError("W must be a finite 6-by-6 matrix.")
    history, cache = [], {}
    best = {"theta": np.array(start, dtype=float), "loss": np.inf}
    def raw_objective(theta):
        key = tuple(np.asarray(theta, dtype=float))
        if key in cache:
            return cache[key]
        try:
            m = inference_mean_moments(theta, ESTIMATION_SEEDS)
            gap = m - np.asarray(target_moments)
            loss = float(gap @ W @ gap)
        except ValueError:
            loss = 1e12
        cache[key] = loss
        if loss < best["loss"]:
            best.update(theta=np.array(theta).copy(), loss=loss)
        history.append(loss)
        if len(history) == 1 or len(history) % 25 == 0:
            print(f"{label}: evaluation {len(history)}, best loss {best['loss']:.6f}", flush=True)
        return loss
    scale = max(raw_objective(start), 1e-8)
    objective = lambda theta: raw_objective(theta) / scale
    exploration = minimize(objective, np.asarray(start, dtype=float), method="Nelder-Mead",
                           bounds=INFERENCE_BOUNDS,
                           options=dict(maxfev=maxfev, xtol=1e-3, ftol=1e-3))
    # Grid-induced discontinuities can cause Powell to return a worse terminal point.
    # Refine the best evaluated point and explicitly check the terminal result.
    simplex = np.tile(best["theta"], (6, 1))
    for j in range(5):
        h = FD_STEPS[j]
        direction = 1 if best["theta"][j] + h <= INFERENCE_BOUNDS[j, 1] else -1
        simplex[j+1, j] += direction * h
    fit = minimize(objective, best["theta"], method="Nelder-Mead", bounds=INFERENCE_BOUNDS,
                   options=dict(maxfev=maxfev, initial_simplex=simplex,
                                xatol=1e-4, fatol=1e-6))
    terminal_loss = raw_objective(fit.x)
    terminal_is_best = abs(terminal_loss - best["loss"]) <= 1e-6 * max(1.0, best["loss"])
    converged = bool(fit.success and terminal_is_best)
    print(f"{label}: converged={converged}; {fit.message}", flush=True)
    if best["loss"] >= 1e12:
        raise ValueError("No valid parameter trial found.")
    return dict(theta=best["theta"], loss=best["loss"], optimizer=fit,
                exploration=exploration, converged=converged, history=history)


def estimate_moment_covariance(theta, seeds=COVARIANCE_SEEDS, cfg=INFERENCE_CONFIG):
    """Covariance of ONE panel's six moments; R is not S in the SE correction."""
    if len(seeds) <= 6 or len(set(seeds)) != len(seeds):
        raise ValueError("Use more than six distinct independent panel seeds.")
    solved = solve_inference_dp(theta, cfg)
    rows = []
    for i, seed in enumerate(seeds, start=1):
        rows.append(inference_panel_moments(theta, solved, seed, cfg))
        if i % 25 == 0:
            print(f"Covariance panels: {i}/{len(seeds)}", flush=True)
    rows = np.asarray(rows)
    Omega = np.cov(rows, rowvar=False, ddof=1)
    # A pseudoinverse can manufacture misleading precision; require invertibility.
    eigenvalues = np.linalg.eigvalsh(Omega)
    if eigenvalues[0] <= 0 or np.linalg.cond(Omega) > 1e12:
        raise ValueError("Moment covariance is singular or ill conditioned; increase panels or diagnose redundant moments.")
    W_star = np.linalg.solve(Omega, np.eye(6))
    return rows, Omega, (W_star + W_star.T) / 2

In [ ]:
def numerical_moment_jacobian(theta, steps=FD_STEPS, seeds=DERIVATIVE_SEEDS):
    """6-by-5 finite difference Jacobian, with common draws and one-sided boundary differences."""
    theta = np.asarray(theta, dtype=float)
    base = inference_mean_moments(theta, seeds)
    D = np.empty((6, 5))
    methods = []
    for j, h in enumerate(steps):
        lower, upper = INFERENCE_BOUNDS[j]
        plus, minus = theta.copy(), theta.copy()
        plus[j] += h
        minus[j] -= h
        if minus[j] >= lower and plus[j] <= upper:
            D[:, j] = (inference_mean_moments(plus, seeds) - inference_mean_moments(minus, seeds)) / (2*h)
            methods.append("central")
        elif plus[j] <= upper:
            D[:, j] = (inference_mean_moments(plus, seeds) - base) / h
            methods.append("forward")
        elif minus[j] >= lower:
            D[:, j] = (base - inference_mean_moments(minus, seeds)) / h
            methods.append("backward")
        else:
            raise ValueError("Finite-difference step is larger than the allowed parameter interval.")
    return D, methods


def standard_errors_from_jacobian(D, W_star, S=S_ESTIMATION):
    """Assignment formula (1+1/S)*(D' W* D)^(-1), with no extra sample-size division."""
    if D.shape != (6, 5) or S < 1:
        raise ValueError("Need a 6-by-5 Jacobian and S >= 1.")
    # With Omega equal to the covariance of panel-level moments, it already
    # contains the finite-panel sampling scale. Do not divide by firms, T, or R again.
    whitened = np.linalg.cholesky(W_star).T @ D
    if np.linalg.matrix_rank(whitened) < 5:
        raise ValueError("Moment Jacobian has rank below 5: ordinary standard errors are not identified on this grid.")
    information = D.T @ W_star @ D
    information = (information + information.T) / 2
    if np.linalg.cond(information) > 1e12:
        raise ValueError("D' W D is too ill conditioned for reliable standard errors.")
    covariance_theta = (1 + 1 / S) * np.linalg.solve(information, np.eye(5))
    covariance_theta = (covariance_theta + covariance_theta.T) / 2
    if np.any(np.diag(covariance_theta) <= 0):
        raise ValueError("Parameter covariance has nonpositive diagonal entries.")
    return covariance_theta, np.sqrt(np.diag(covariance_theta))

In [ ]:
theta_start_inference = np.array([alpha, gamma, rho, sigma, phi_tilde0], dtype=float)
stage1 = estimate_inference_stage(theta_start_inference, np.eye(6), "Identity stage")
theta_stage1 = stage1["theta"]
print("Stage-1 estimates:", dict(zip(PARAMETER_NAMES, theta_stage1)))
print("Stage-1 converged:", stage1["converged"])